# LexisNexis Fraud Detection - Task 5: Hyperparameter Tuning

Tunes the key settings of each model (logistic regression, random forest, gradient boosting) to get the best PR-AUC. Setup (data, features, preprocessing) is copied from Task 1, including `NewPolicy`, so every model sees the same 129 features.

In [10]:
!git clone -q https://github.com/Break-Through-Tech/LexisNexis-1A-detecting-fraudulent-insurance-claims.git
%cd /content/LexisNexis-1A-detecting-fraudulent-insurance-claims

/content/LexisNexis-1A-detecting-fraudulent-insurance-claims


In [11]:
import json

import numpy as np
import pandas as pd
from scipy.stats import loguniform

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

pd.set_option("display.max_columns", None)

In [12]:
RANDOM_STATE = 42
N_ITER = 30      # settings combinations tried per model
CV_FOLDS = 5     # cross-validation folds inside 1994
N_BOOT = 1000    # bootstrap resamples for the noise check at the end

In [13]:
df = pd.read_csv("data/fraud_oracle_cleaned.csv")
TARGET = "FraudFound_P"
SAFE_FEATURES = [
    "Month", "WeekOfMonth", "DayOfWeek", "MonthClaimed", "WeekOfMonthClaimed",
    "DayOfWeekClaimed", "Make", "VehicleCategory", "VehiclePrice", "AgeOfVehicle",
    "NumberOfCars", "AccidentArea", "Fault", "Sex", "MaritalStatus", "Age",
    "Age_missing", "AgeOfPolicyHolder", "PolicyType", "BasePolicy", "Deductible",
    "DriverRating", "AgentType", "Days_Policy_Accident", "Days_Policy_Claim",
    "PastNumberOfClaims",
]

train = df[df["Year"] == 1994].copy()
val = df[df["Year"] == 1995].copy()
test = df[df["Year"] == 1996].copy()   # not used in this notebook

MONTH_NUM = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
             "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}
VEHICLE_AGE_MAP = {"new": 0, "2 years": 2, "3 years": 3, "4 years": 4,
                   "5 years": 5, "6 years": 6, "7 years": 7, "more than 7": 8}

def add_engineered_features(data):
    data = data.copy()
    data["AgeBand"] = pd.cut(
        data["Age"], bins=[0, 24, 34, 44, 54, 64, np.inf],
        labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"])
    data["AgeBand"] = data["AgeBand"].astype("object").fillna("Missing")
    data["VehicleAgeApprox"] = data["AgeOfVehicle"].map(VEHICLE_AGE_MAP)
    accident_month = data["Month"].map(MONTH_NUM)
    claimed_month = data["MonthClaimed"].map(MONTH_NUM)
    data["ClaimLagMonths"] = (claimed_month - accident_month) % 12
    data["NewPolicy"] = data["Days_Policy_Accident"].isin(["1 to 7", "8 to 15", "15 to 30"]).astype(int)
    data["WeekendAccident"] = data["DayOfWeek"].isin(["Saturday", "Sunday"]).astype(int)
    data["WeekendClaim"] = data["DayOfWeekClaimed"].isin(["Saturday", "Sunday"]).astype(int)
    return data

X_train = add_engineered_features(train[SAFE_FEATURES])
X_val = add_engineered_features(val[SAFE_FEATURES])
y_train, y_val = train[TARGET], val[TARGET]

print("Train (1994):", X_train.shape, "fraud cases:", y_train.sum())
print("Validation (1995):", X_val.shape, "fraud cases:", y_val.sum())

Train (1994): (6141, 32) fraud cases: 409
Validation (1995): (5195, 32) fraud cases: 301


In [14]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()
grouped_categories = ["Make", "PolicyType"]
regular_categories = [c for c in categorical_features if c not in grouped_categories]

def make_preprocessor(dense=False):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    grouped_category_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01)),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, regular_categories),
        ("grouped_categorical", grouped_category_pipeline, grouped_categories),
    ], sparse_threshold=0 if dense else 0.3)

def build_pipeline(model, dense=False):
    """Preprocessing + model in one object, so .fit() always fits preprocessing on the training data only."""
    return Pipeline([("prep", make_preprocessor(dense)), ("clf", model)])

print("Processed features:", make_preprocessor().fit(X_train).transform(X_train).shape[1])  # should be 129

Processed features: 129


## Evaluation function (same as Tasks 2 and 3)

Kept identical so the rows here line up with the earlier notebooks: precision, recall and F1 at a 0.5 threshold, PR-AUC, and the share of fraud captured in the top 10% / 20% highest-scored claims.

In [15]:
def evaluate(name, model, X_tr, y_tr, X_ev, y_ev, top_fracs=(0.10, 0.20)):
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_ev)[:, 1]
    pred = (proba >= 0.5).astype(int)
    row = {
        "Approach": name,
        "Precision": precision_score(y_ev, pred, zero_division=0),
        "Recall": recall_score(y_ev, pred),
        "F1": f1_score(y_ev, pred),
        "PR-AUC": average_precision_score(y_ev, proba),
    }
    y_sorted = y_ev.to_numpy()[proba.argsort()[::-1]]
    for f in top_fracs:
        k = int(len(y_sorted) * f)
        row[f"Fraud captured top {int(f*100)}%"] = y_sorted[:k].sum() / y_sorted.sum()
    return row

## How the tuning works

For each model, RandomizedSearchCV picks N_ITER random combinations of settings from the ranges defined below. Each combination is scored with 5-fold cross-validation on 1994: the 1994 claims are split into 5 chunks (each keeping the 6.7% fraud rate), the model trains on 4 chunks and gets scored on the 5th, and this rotates until every chunk has been scored once. The 5 scores are averaged, and the combination with the best average PR-AUC wins. Top 10% and top 20% fraud capture are tracked too.

In [16]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def capture_scorer(frac):
    """Scorer for the business metric: share of all fraud found in the top `frac` highest-scored claims."""
    def score(estimator, X, y):
        proba = estimator.predict_proba(X)[:, 1]
        y_arr = np.asarray(y)
        k = int(len(y_arr) * frac)
        top = np.argsort(proba)[::-1][:k]
        return y_arr[top].sum() / y_arr.sum()
    return score

SCORING = {
    "pr_auc": "average_precision",   # scikit-learn's name for PR-AUC
    "top10": capture_scorer(0.10),
    "top20": capture_scorer(0.20),
}

def clean_params(params):
    """Strip the 'clf__' prefix and turn numpy numbers into plain Python numbers."""
    return {k.replace("clf__", ""): (v.item() if hasattr(v, "item") else v) for k, v in params.items()}

def run_search(name, model, param_space, dense=False):
    search = RandomizedSearchCV(
        build_pipeline(model, dense),
        param_distributions=param_space,
        n_iter=N_ITER,
        scoring=SCORING,
        refit="pr_auc",              # pick the winner by PR-AUC
        cv=cv,
        return_train_score=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)     # 1994 only
    spread = search.cv_results_["std_test_pr_auc"][search.best_index_]
    print(f"{name}: best CV PR-AUC {search.best_score_:.4f} (+/- {spread:.4f} across folds)")
    print("Best settings:", clean_params(search.best_params_))
    return search

def show_top(search, k=5):
    """The k best settings combinations, ranked by cross-validated PR-AUC."""
    res = pd.DataFrame(search.cv_results_)
    params = [c for c in res.columns if c.startswith("param_")]
    cols = ["mean_test_pr_auc", "std_test_pr_auc", "mean_train_pr_auc",
            "mean_test_top10", "mean_test_top20"] + params
    out = res.sort_values("mean_test_pr_auc", ascending=False)[cols].head(k)
    out.columns = [c.replace("param_clf__", "") for c in out.columns]
    return out.round(4).reset_index(drop=True)

## 1. Logistic regression

The main changer is C, which controls regularization. A small C pulls the weights toward zero which means we'd have a simpler model and less overfitting, and a large C lets the weights grow to fit 1994 closely. It's searched on a log scale from 0.001 to 100, since its effect multiplies.

In [17]:
lr_space = {"clf__C": loguniform(1e-3, 1e2)}

lr_search = run_search(
    "Logistic regression",
    LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE),
    lr_space,
)
show_top(lr_search)

Logistic regression: best CV PR-AUC 0.2458 (+/- 0.0262 across folds)
Best settings: {'C': 0.009962513222055111}


,mean_test_pr_auc,std_test_pr_auc,mean_train_pr_auc,mean_test_top10,mean_test_top20,C
0,0.2458,0.0262,0.2744,0.4032,0.6600,0.0100
1,0.2458,0.0259,0.2753,0.4032,0.6624,0.0115
2,0.2456,0.0253,0.2801,0.3910,0.6624,0.0332
3,0.2455,0.0246,0.2796,0.3885,0.6624,0.0289
4,0.2454,0.0246,0.2797,0.3885,0.6600,0.0286


## 2. Random Forest


We will use `GridSearchCV` to tune key hyperparameters of the Random Forest. We'll optimize for `average_precision` (PR-AUC) since it is a robust metric for highly imbalanced fraud detection tasks.

In [18]:
from sklearn.model_selection import GridSearchCV

rf_grid = {
    "clf__n_estimators": [300, 500],
    "clf__max_depth": [10, 20, None],
    "clf__min_samples_leaf": [10, 30, 50, 100],
    "clf__max_features": ["sqrt", "log2"],
}

rf_search = GridSearchCV(
    build_pipeline(RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1)),
    param_grid=rf_grid,
    scoring=SCORING,
    refit="pr_auc",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)   # raw 1994 features; the pipeline handles preprocessing

print(f"Random forest: best CV PR-AUC {rf_search.best_score_:.4f}")
print("Best settings:", clean_params(rf_search.best_params_))
show_top(rf_search)

Random forest: best CV PR-AUC 0.2941
Best settings: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'n_estimators': 500}


,mean_test_pr_auc,std_test_pr_auc,mean_train_pr_auc,mean_test_top10,mean_test_top20,max_depth,max_features,min_samples_leaf,n_estimators
0,0.2941,0.0397,0.6171,0.4204,0.6600,None,sqrt,10,500
1,0.2926,0.0400,0.6148,0.4155,0.6649,None,sqrt,10,300
2,0.2919,0.0399,0.6144,0.4229,0.6600,20,sqrt,10,500
3,0.2911,0.0390,0.6050,0.4229,0.6648,20,log2,10,500
4,0.2907,0.0449,0.6059,0.4204,0.6575,None,log2,10,500


## 3. Gradient boosting
(we dont have to do this, I did it anyways)
- learning_rate: how big a correction each new tree makes. Smaller values need more trees but usually generalize better.
- max_iter: number of trees. Tuned together with learning_rate.
- max_leaf_nodes, max_depth: size of each tree.
- min_samples_leaf: same idea as in the random forest.
- l2_regularization: a penalty that keeps each tree's corrections small.
- max_features: share of columns each split can consider.

early_stopping=False so max_iter is the real number of trees.

In [19]:
xgb_search = None
gb_space = {
    "clf__learning_rate": loguniform(0.01, 0.2),
    "clf__max_iter": [100, 200, 400, 600],
    "clf__max_leaf_nodes": [7, 15, 31],
    "clf__max_depth": [None, 3, 5],
    "clf__min_samples_leaf": [20, 50, 100, 200],
    "clf__l2_regularization": [0.0, 0.1, 1.0, 10.0],
    "clf__max_features": [0.3, 0.5, 1.0],
}

gb_search = run_search(
    "Gradient boosting",
    HistGradientBoostingClassifier(class_weight="balanced", early_stopping=False,
                                   random_state=RANDOM_STATE),
    gb_space,
    dense=True,
)
show_top(gb_search)

Gradient boosting: best CV PR-AUC 0.3020 (+/- 0.0379 across folds)
Best settings: {'l2_regularization': 0.1, 'learning_rate': 0.029266761285490727, 'max_depth': None, 'max_features': 1.0, 'max_iter': 100, 'max_leaf_nodes': 15, 'min_samples_leaf': 20}


,mean_test_pr_auc,std_test_pr_auc,mean_train_pr_auc,mean_test_top10,mean_test_top20,l2_regularization,learning_rate,max_depth,max_features,max_iter,max_leaf_nodes,min_samples_leaf
0,0.3020,0.0379,0.4463,0.4375,0.6942,0.1,0.0293,None,1.0,100,15,20
1,0.2859,0.0327,0.9497,0.4107,0.6796,1.0,0.0411,None,1.0,200,31,20
2,0.2778,0.0435,0.4068,0.4228,0.6770,10.0,0.0834,3,0.3,200,15,50
3,0.2772,0.0349,0.6477,0.3959,0.6599,1.0,0.0831,5,1.0,100,31,20
4,0.2754,0.0299,0.6011,0.4155,0.6697,0.1,0.0100,None,0.3,400,31,50


## Results on the 1995 validation set

Each model is retrained on all of 1994 and scored once on 1995 with the team's `evaluate()`. The untuned reference rows should match Task 3's table (LR + class_weight PR-AUC 0.0948, RF with Task 3 settings 0.1098, RF from Task 3's grid search 0.1100). If they don't, something in your setup differs from the team's.

In [20]:
candidates = {
    # Untuned references from earlier tasks
    "LR + class_weight (untuned, Task 2)": build_pipeline(
        LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    "RF (Task 3 settings)": build_pipeline(
        RandomForestClassifier(n_estimators=500, min_samples_leaf=20, class_weight="balanced",
                               random_state=RANDOM_STATE, n_jobs=-1)),
    "RF (Task 3 grid search)": build_pipeline(
        RandomForestClassifier(n_estimators=300, min_samples_leaf=30, max_features="sqrt",
                               class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
    "GB (default settings)": build_pipeline(
        HistGradientBoostingClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        dense=True),
    # Tuned in this notebook
    "LR tuned": lr_search.best_estimator_,
    "RF tuned": rf_search.best_estimator_,
    "GB tuned": gb_search.best_estimator_,
}
if xgb_search is not None:
    candidates["XGBoost tuned"] = xgb_search.best_estimator_

results, val_probs = [], {}
for name, model in candidates.items():
    results.append(evaluate(name, model, X_train, y_train, X_val, y_val))
    val_probs[name] = model.predict_proba(X_val)[:, 1]   # evaluate() just fit it on 1994

results_df = pd.DataFrame(results).set_index("Approach").round(4)
results_df

,Precision,Recall,F1,PR-AUC,Fraud captured top 10%,Fraud captured top 20%
Approach,,,,,,
"LR + class_weight (untuned, Task 2)",0.1009,0.4319,0.1636,0.0948,0.1927,0.3389
RF (Task 3 settings),0.1158,0.4352,0.1830,0.1098,0.2027,0.3821
RF (Task 3 grid search),0.1170,0.5050,0.1900,0.1100,0.2027,0.4053
GB (default settings),0.1050,0.2093,0.1398,0.1184,0.1960,0.3920
LR tuned,0.1093,0.5282,0.1811,0.1017,0.1728,0.3455
RF tuned,0.1144,0.2857,0.1633,0.1099,0.1993,0.3887
GB tuned,0.1098,0.5150,0.1810,0.1417,0.2359,0.4053


## Noise check: are the differences real?

1995 has only about 300 fraud cases, so PR-AUC moves around by chance. This cell resamples the 1995 claims 1,000 times (a bootstrap) and recomputes PR-AUC on each resample.

- 95% low / 95% high: the range PR-AUC falls in for 95% of resamples.
- Diff vs LR: PR-AUC minus the untuned logistic regression's PR-AUC, with its own 95% range.
- Beats LR: share of resamples where the model beat the untuned logistic regression.

If the 95% range of the difference includes 0, we can't confidently call that model better.

In [21]:
BASELINE = "LR + class_weight (untuned, Task 2)"
rng = np.random.default_rng(RANDOM_STATE)
y_val_arr = y_val.to_numpy()
boot_idx = rng.integers(0, len(y_val_arr), size=(N_BOOT, len(y_val_arr)))

def boot_pr_auc(proba):
    return np.array([average_precision_score(y_val_arr[i], proba[i]) for i in boot_idx])

base_point = average_precision_score(y_val_arr, val_probs[BASELINE])
base_boot = boot_pr_auc(val_probs[BASELINE])

rows = []
for name, proba in val_probs.items():
    point = average_precision_score(y_val_arr, proba)
    s = boot_pr_auc(proba)
    d = s - base_boot
    is_base = name == BASELINE
    rows.append({
        "Approach": name,
        "PR-AUC": point,
        "95% low": np.percentile(s, 2.5),
        "95% high": np.percentile(s, 97.5),
        "Diff vs LR": np.nan if is_base else point - base_point,
        "Diff 95% low": np.nan if is_base else np.percentile(d, 2.5),
        "Diff 95% high": np.nan if is_base else np.percentile(d, 97.5),
        "Beats LR": np.nan if is_base else (d > 0).mean(),
    })

noise_df = pd.DataFrame(rows).set_index("Approach").round(4)
noise_df

,PR-AUC,95% low,95% high,Diff vs LR,Diff 95% low,Diff 95% high,Beats LR
Approach,,,,,,,
"LR + class_weight (untuned, Task 2)",0.0948,0.0809,0.1132,NaN,NaN,NaN,NaN
RF (Task 3 settings),0.1098,0.0956,0.1272,0.0150,0.0051,0.0232,0.997
RF (Task 3 grid search),0.1100,0.0959,0.1278,0.0152,0.0051,0.0235,0.997
GB (default settings),0.1184,0.0944,0.1451,0.0236,0.0011,0.0483,0.985
LR tuned,0.1017,0.0886,0.1183,0.0068,0.0001,0.0117,0.976
RF tuned,0.1099,0.0954,0.1271,0.0151,0.0051,0.0235,0.997
GB tuned,0.1417,0.1112,0.1745,0.0469,0.0206,0.0762,1.000


## Save results for Task 6


In [22]:
searches = {"logistic_regression": lr_search, "random_forest": rf_search, "gradient_boosting": gb_search}
if xgb_search is not None:
    searches["xgboost"] = xgb_search

tuned = {
    name: {"best_params": clean_params(s.best_params_), "cv_pr_auc_1994": round(float(s.best_score_), 4)}
    for name, s in searches.items()
}

with open("m2_task5_tuned_params.json", "w") as f:
    json.dump(tuned, f, indent=2)
results_df.join(noise_df.drop(columns="PR-AUC")).to_csv("m2_task5_validation_results.csv")

print(json.dumps(tuned, indent=2))
print("\nRanked by validation PR-AUC:")
print(results_df.sort_values("PR-AUC", ascending=False)[["PR-AUC", "Fraud captured top 10%", "Fraud captured top 20%"]])

{
  "logistic_regression": {
    "best_params": {
      "C": 0.009962513222055111
    },
    "cv_pr_auc_1994": 0.2458
  },
  "random_forest": {
    "best_params": {
      "max_depth": null,
      "max_features": "sqrt",
      "min_samples_leaf": 10,
      "n_estimators": 500
    },
    "cv_pr_auc_1994": 0.2941
  },
  "gradient_boosting": {
    "best_params": {
      "l2_regularization": 0.1,
      "learning_rate": 0.029266761285490727,
      "max_depth": null,
      "max_features": 1.0,
      "max_iter": 100,
      "max_leaf_nodes": 15,
      "min_samples_leaf": 20
    },
    "cv_pr_auc_1994": 0.302
  }
}

Ranked by validation PR-AUC:
                                     PR-AUC  Fraud captured top 10%  \
Approach                                                              
GB tuned                             0.1417                  0.2359   
GB (default settings)                0.1184                  0.1960   
RF (Task 3 grid search)              0.1100                  0.2027   
RF

## Summary

We tuned logistic regression and gradient boosting with randomized search (30 settings combinations each) and the random forest with a grid search (48 combinations), scoring each combination by 5-fold cross-validated PR-AUC inside the 1994 training set. Preprocessing ran inside each pipeline, class weighting was applied to every model, and the 1996 test set was not used. The untuned reference rows reproduce Task 3's numbers exactly, so this setup matches the rest of the team's notebooks.

On the 1995 validation set, the best model was tuned gradient boosting with a PR-AUC of 0.1417 (untuned logistic regression: 0.0948), capturing 40.5% of fraud in the top 20% of scored claims (untuned logistic regression: 33.9%) and 23.6% in the top 10% (untuned logistic regression: 19.3%). Tuning moved random forest PR-AUC from 0.1098 (Task 3 settings) to 0.1099, so the forest gained essentially nothing, and moved gradient boosting from 0.1184 (default settings) to 0.1417. Tuned logistic regression improved slightly, from 0.0948 to 0.1017, using much stronger regularization (C ≈ 0.01) than the default (C = 1).

The bootstrap check shows every model beats untuned logistic regression: the 95% range of each difference stays above 0. Tuned gradient boosting has the clearest gain (+0.047, 95% range +0.021 to +0.076, ahead of logistic regression in all 1,000 resamples). Tuned logistic regression only barely clears 0 (95% range +0.0001 to +0.0117). The check compares each model to logistic regression only, so it doesn't directly test gradient boosting against the random forests. Their individual 95% ranges overlap (0.111 to 0.175 for tuned gradient boosting, about 0.095 to 0.128 for the forests).

One limitation: cross-validated PR-AUC inside 1994 (0.25 to 0.30) runs far above the 1995 validation PR-AUC (0.10 to 0.14). Shuffled folds within one year are easier than predicting the next year, so cross-validation overstates real performance. The random forest shows this most clearly. Its cross-validation winner used the smallest leaves tried (min_samples_leaf = 10), with a training PR-AUC of 0.62 against 0.29 in cross-validation, a sign of memorizing 1994. Those settings did not improve on 1995, and recall at the 0.5 threshold dropped to 0.286.

Suggestion for Task 6: carry tuned gradient boosting forward as the lead model. For the random forest comparison, the Task 3 grid version (min_samples_leaf = 30) matches the tuned forest's PR-AUC with much better recall at the 0.5 threshold (0.505 vs 0.286).

Best settings for each model are saved in `m2_task5_tuned_params.json` for Task 6.